# Fine Tunning BERTax Cacao

Instalar paquetes necesarios

In [1]:
!pip install biopython
!pip install keras-pos-embd
!pip install keras_layer_normalization
!pip install keras_transformer
!pip install keras-bert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for keras-pos-embd: filename=keras_pos_embd-0.13.0-py3-none-any.whl size=6945 sha256=ea6c2dfeab56eaedda32e643b96d362c10fe340c473de4d6e637c7b365ea9ef7
  Stored in directory: /root/.cache/pip/wheels/f3/b3/f2/9acf9a5c6b16a27d3e9080901f49ae2774dcaca251a752d82a
Successfully built keras-pos-embd
  Preparing metadata (setup.py) ... done
  Created wheel for keras_layer_normalization: filename=keras_layer_normalization-0.16.0-py3-none-any.whl size=4654 sha256=95b7fa2e2449e3d1a0c40c20545ee0d837396f445a4bc5c81fe925c4978ee459
  Stored in directory: /root/.cache/pip/wheels/0f/c4/a1/24f1ca7fd39e75f4d8dab7feda6fe3e2163d8062b29f1169fb
Successfully built keras_layer_normalization
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py

In [3]:
import tensorflow as tf
from tensorflow import keras
import json
from preprocessing.process_inputs import get_class_vectors, ALPHABET
from model import PARAMS
import numpy as np
from tensorflow.keras.utils import Sequence
from bert_utils import get_token_dict, seq2tokens, predict
from bert_utils import generate_bert_with_pretrained, generate_bert_with_pretrained_multi_tax, \
    get_classes_and_weights_multi_tax
from random import shuffle, sample
from sklearn.model_selection import train_test_split
import os
import argparse
from dataclasses import dataclass, field
from typing import List, Optional
from logging import warning
import pickle
from tensorflow.keras import mixed_precision
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard, EarlyStopping
from tensorflow.keras.callbacks import Callback
from os.path import splitext
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from keras.models import load_model
from dependencies.keras_bert.keras_bert.loader import build_model_from_config

In [ ]:
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

Num GPUs Available:  1


In [4]:
class LossHistoryLogger(Callback):
    def __init__(self, filename='loss_history_fine_tune.csv'):
        super().__init__()
        self.filename = filename
        self.history = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        logs['epoch'] = epoch + 1
        self.history.append(logs)
        df = pd.DataFrame(self.history)
        df.to_csv(self.filename, index=False)

In [5]:
def load_dataset(filepath):
    df = pd.read_csv(filepath, sep="\t")
    x = df["x"]
    y = df["y"]
    y_species = df["tax_id"]
    return x, y, y_species

In [6]:
def load_fragments(fragments_dir, shuffle_=True, balance=True, nr_seqs=None):
    fragments = []
    species_list = []
    for class_ in classes:
        fragments.append((class_, json.load(open(os.path.join(
            fragments_dir, f'{class_}_fragments.json')))))
        species_list.append([int(line.strip()) for line in
                             open(os.path.join(fragments_dir, f'{class_}_species_picked.txt')).readlines()])
    nr_seqs_max = min(len(item[1]) for item in fragments)
    if (nr_seqs is None or nr_seqs > nr_seqs_max):
        nr_seqs = nr_seqs_max
    x = []
    y = np.array([])
    y_species = np.array([], dtype=int)

    for index, fragments_i in enumerate(fragments):
        class_, class_fragments = fragments_i
        if not balance:
            x.extend(class_fragments)
            y = np.append(y, [class_] * len(class_fragments))
            y_species = np.append(y_species, species_list[index])
        else:
            x_help = list(zip(class_fragments, species_list[index]))
            # x.extend(sample(class_fragments, nr_seqs))
            x_help = sample(x_help, nr_seqs)
            x_help, y_species_help = zip(*x_help)
            x.extend(x_help)
            y_species = np.append(y_species, y_species_help)
            y = np.append(y, [class_] * nr_seqs)

    assert len(x) == len(y)
    if (shuffle_):
        to_shuffle = list(zip(x, y, y_species))
        shuffle(to_shuffle)
        x, y, y_species = zip(*to_shuffle)
    # SOBREESCRIBIR LAS ETIQUETAS BASÁNDONOS EN tax_id:
    y = np.array(["Cacao" if taxid == 3641 else "NotCacao" for taxid in y_species])

    print(f'{len(x)} fragments loaded in total; '
          f'balanced={balance}, shuffle_={shuffle_}, nr_seqs={nr_seqs}')
    return np.array(x), np.array(y), np.array(y_species)

In [7]:
@dataclass
class FragmentGenerator(Sequence):
    x: list
    y: list
    seq_len: int
    max_seq_len: Optional[int] = None
    k: int = 3
    stride: int = 3
    batch_size: int = 32
    classes: List = field(default_factory=lambda:
    ["NotCacao", "Cacao"])
    seq_len_like: Optional[np.array] = None
    window: bool = False

    def __post_init__(self):
        self.class_vectors = get_class_vectors(self.classes)
        self.token_dict = get_token_dict(ALPHABET, k=3)
        if (self.max_seq_len is None):
            self.max_seq_len = self.seq_len

    def __len__(self):
        return np.ceil(len(self.x)
                       / float(self.batch_size)).astype(int)

    def __getitem__(self, idx):
      batch_fragments = self.x[idx * self.batch_size:
                              (idx + 1) * self.batch_size]
      batch_x = [seq2tokens(seq, self.token_dict, seq_length=self.seq_len,
                            max_length=self.max_seq_len,
                            k=self.k, stride=self.stride, window=self.window,
                            seq_len_like=self.seq_len_like)
                for seq in batch_fragments]

      x_inputs = (
          np.array([item[0] for item in batch_x]),  # Token input
          np.array([item[1] for item in batch_x])   # Segment input
      )

      if self.y is not None and len(self.y) != 0:
          batch_classes = self.y[idx * self.batch_size:
                                (idx + 1) * self.batch_size]
          batch_y = np.array([self.class_vectors[c] for c in batch_classes])
          return x_inputs, batch_y  # (inputs, labels)
      else:
          return x_inputs  # solo inputs, aún como tupla

In [8]:
def get_fine_model(pretrained_model_file, weights_path=None):
    # with mirrored_strategy.scope():
    model_fine = generate_bert_with_pretrained(
        pretrained_model_file, len(classes), weights_path=weights_path)
    model_fine.compile(keras.optimizers.Adam(learning_rate),
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])
    max_length = model_fine.input_shape[0][1]
    return model_fine, max_length

In [9]:
!unzip bert_nc_trained.keras model.weights.h5
!unzip bert_nc_trained.keras config.json

Archive:  bert_nc_trained.keras
 extracting: model.weights.h5        
Archive:  bert_nc_trained.keras
 extracting: config.json             


In [12]:
pretrained_bert= '/content/config.json'
fragments_dir = '/content/3-4_input_bert_cn'
weights_path='/content/model.weights.h5'
seq_len = 151
seq_len_like_path = None  # 'path/to/seq_len_like.pkl'
k = 3
stride = 3
batch_size = 64
epochs = 10
nr_seqs = 7903874
learning_rate = 5.0973e-5
save_name = 'fine-tune-BERTaxCacao'
store_predictions = True
store_train_data = False
roc_auc = False
multi_tax = False
tax_ranks = ["Cacao", "NotCacao"]
only_test_model = False
use_defined_train_test_set = False
test = only_test_model
norm_weights = True
classes = ["NotCacao", "Cacao"]
seq_len_like = None

In [13]:
x, y, y_species = load_fragments(fragments_dir, nr_seqs=nr_seqs)
f_train_x, f_test_x, f_train_y, f_test_y = train_test_split(x, y, test_size=0.2, stratify=y)
f_train_x, f_val_x, f_train_y, f_val_y = train_test_split(f_train_x, f_train_y, test_size=0.05, stratify=f_train_y)

model, max_length = get_fine_model(pretrained_bert, weights_path=weights_path)
if seq_len > max_length:
    warning(f'desired seq len ({seq_len}) is higher than possible ({max_length}), setting to {max_length}')
    seq_len = max_length

generator_args = {
    'max_seq_len': max_length, 'k': k, 'stride': stride,
    'batch_size': batch_size, 'window': True,
    'seq_len_like': seq_len_like
}

model.summary()

name = save_name if save_name else ""
save_path = save_name if save_name else os.path.splitext(pretrained_bert)[0]
filepath1 = save_path + "_model.best.acc.keras"
filepath2 = save_path + "_model.best.loss.keras"

checkpoint1 = ModelCheckpoint(filepath1, monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')
checkpoint2 = ModelCheckpoint(filepath2, monitor='val_loss', verbose=1, save_best_only=True, mode='min')
checkpoint3 = EarlyStopping('val_loss', patience=2, restore_best_weights=True)

tensorboard_callback = TensorBoard(log_dir="./logs/run_broad_learning" + name, histogram_freq=1)

#callbacks_list = [checkpoint1, checkpoint2, checkpoint3]
callbacks_list = [checkpoint1, checkpoint2, checkpoint3, tensorboard_callback, LossHistoryLogger(filename=name + '_loss_history.csv')]

if store_train_data:
    from datetime import datetime
    time_str = datetime.now().strftime('%Y%m%d_%H-%M-%S')
    for kind, x, y in [('train', f_train_x, f_train_y), ('val', f_val_x, f_val_y), ('test', f_test_x, f_test_y)]:
        with open(f'{time_str}_{kind}_data.json', 'w') as f:
            json.dump([x, y], f)
    print('saved train/test/val data.')

test_g = FragmentGenerator(f_test_x, f_test_y, seq_len, **generator_args)

try:
    model.fit(
        FragmentGenerator(f_train_x, f_train_y, seq_len, **generator_args),
        callbacks=callbacks_list, epochs=epochs,
        validation_data=FragmentGenerator(f_val_x, f_val_y, seq_len, **generator_args),
        verbose=1)
except KeyboardInterrupt:
    print("Training interrupted. Saving model...")
    model.save(splitext(pretrained_bert)[0] + '_aborted.keras')
    print('Testing interrupted model...')
    result = model.evaluate(test_g)
    print("Test results:", *zip(model.metrics_names, result))
    exit()

final_save_path = save_name + '.keras' if save_name else os.path.splitext(pretrained_bert)[0] + '_finetuned.keras'
model.save(final_save_path)
print('Testing final model...')

if store_predictions or roc_auc:
    predicted = predict(
        model, test_g,
        roc_auc, classes, return_data=store_predictions, calc_metrics=False)

    y_true, y_pred = predicted["data"]
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Convertir one-hot a etiquetas si es necesario
    if len(y_true.shape) > 1:
        y_true = np.argmax(y_true, axis=1)
    if len(y_pred.shape) > 1:
        y_pred = np.argmax(y_pred, axis=1)

    acc = balanced_accuracy_score(y_true, y_pred)

    conf_matrix = pd.crosstab(
        y_true, y_pred,
        rownames=['True'],
        colnames=['Predicted'],
        margins=True
    )

    #if verbose:
    print(f"Balanced Accuracy: {acc:.4f}")
    print("Matriz de confusión:")
    print(conf_matrix)

    result = predicted['metrics']
    metrics_names = predicted['metrics_names']

    if store_predictions:
        filename = "test_predictions.pkl" if test else os.path.splitext(final_save_path)[0] + '_predictions.pkl'
        with open(filename, 'wb') as f:
            pickle.dump(predicted, f)
else:
    result = model.evaluate(test_g)
    metrics_names = model.metrics_names

print("Test results:", *zip(metrics_names, result))

703444 fragments loaded in total; balanced=True, shuffle_=True, nr_seqs=351722
Ana test :: inputs: 
Tensor("Placeholder:0", shape=(None, 151, 256), dtype=float32)


/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:629: UserWarning: A total of 3 objects could not be loaded. Example error message for object <EmbeddingSimilarity name=MLM-Sim, built=True>:

Layer 'MLM-Sim' expected 1 variables, but received 0 variables during loading. Expected: ['bias']

List of objects that could not be loaded:
[<EmbeddingSimilarity name=MLM-Sim, built=True>, <Dense name=NSP-Dense, built=True>, <Dense name=NSP, built=True>]
  warnings.warn(msg)


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Input-Token (InputLayer)  │ (None, 151)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Input-Segment             │ (None, 151)            │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Embedding-Token           │ (None, 151, 256)       │         17,664 │ Input-Token[0][0]      │
│ (TokenEmbedding)          │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Embedding-Segment         │ (None, 151, 256)       │            512 │ Input-Segment[0][0]    │
│ (Embedding)               │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_1 (Add)               │ (None, 151, 256)       │              0 │ Embedding-Token[0][0], │
│                           │                        │                │ Embedding-Segment[0][… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Embedding-Position        │ (None, 151, 256)       │         38,656 │ add_1[0][0]            │
│ (PositionEmbedding)       │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Embedding-Dropout         │ (None, 151, 256)       │              0 │ Embedding-Position[0]… │
│ (Dropout)                 │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Embedding-Norm            │ (None, 151, 256)       │            512 │ Embedding-Dropout[0][… │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Encoder-1-MultiHeadSelfA… │ (None, 151, 256)       │        263,168 │ Embedding-Norm[0][0]   │
│ (MultiHeadAttention)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Encoder-1-MultiHeadSelfA… │ (None, 151, 256)       │              0 │ Encoder-1-MultiHeadSe… │
│ (Dropout)                 │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Encoder-1-MultiHeadSelfA… │ (None, 151, 256)       │              0 │ Embedding-Norm[0][0],  │
│ (Add)                     │                        │                │ Encoder-1-MultiHeadSe… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Encoder-1-MultiHeadSelfA… │ (None, 151, 256)       │            512 │ Encoder-1-MultiHeadSe… │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Encoder-1-FeedForward     │ (None, 151, 256)       │      1,050,880 │ Encoder-1-MultiHeadSe… │
│ (FeedForward)             │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ Encoder-1-FeedForward

 Total params: 9,329,154 (35.59 MB)

 Trainable params: 9,329,154 (35.59 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


8354/8354 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.9512 - loss: 0.1473
Epoch 1: val_accuracy improved from -inf to 0.96997, saving model to fine-tune-BERTaxCacao_model.best.acc.keras

Epoch 1: val_loss improved from inf to 0.10968, saving model to fine-tune-BERTaxCacao_model.best.loss.keras
8354/8354 ━━━━━━━━━━━━━━━━━━━━ 712s 80ms/step - accuracy: 0.9512 - loss: 0.1473 - val_accuracy: 0.9700 - val_loss: 0.1097 - epoch: 1.0000
Epoch 2/10
8354/8354 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.9753 - loss: 0.0875
Epoch 2: val_accuracy improved from 0.96997 to 0.97946, saving model to fine-tune-BERTaxCacao_model.best.acc.keras

Epoch 2: val_loss improved from 0.10968 to 0.07634, saving model to fine-tune-BERTaxCacao_model.best.loss.keras
8354/8354 ━━━━━━━━━━━━━━━━━━━━ 624s 75ms/step - accuracy: 0.9753 - loss: 0.0875 - val_accuracy: 0.9795 - val_loss: 0.0763 - epoch: 2.0000
Epoch 3/10
8354/8354 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.9797 - loss: 0.0735
Epoch 3: val_accura